# Hafta 8 — Denetimsiz Öğrenme: k-means, Hiyerarşik Kümeleme, PCA

Veri: `abone_profilleri.csv` — 300 abone × 24 saatlik ortalama yük (kW). Etiket yok; amaç tipik profil gruplarını bulmak.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage

ab = pd.read_csv("abone_profilleri.csv")
P = ab.drop(columns="abone_id").values; h = np.arange(24)
Pn = P / P.mean(1, keepdims=True)                 # şekil: her profilin ortalaması 1
sc = StandardScaler().fit(Pn); Z = sc.transform(Pn)
print(ab.shape); ab.head()

## 1. Veriye bakalım: ham ve normalize profiller

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for i in range(15): ax[0].plot(h, P[i], lw=1); ax[1].plot(h, Pn[i], lw=1)
ax[0].set_title("Ham (kW) — büyüklük farkı baskın"); ax[1].set_title("Normalize — şekil görünür"); ax[0].set_xlabel("saat"); ax[1].set_xlabel("saat"); plt.show()
print("Ham profil ortalamaları (kW):", P.mean(1).min().round(1), "-", P.mean(1).max().round(1))

**Soru:** Neden ham kW değil normalize profil kümeliyoruz? Ham veriyle kümelesek kümeler neye göre oluşurdu?

## 2. k-means elle (Örnek 8.1)

In [ ]:
x = np.array([2, 3, 4, 10, 11, 25.])
def kmeans1d(x, m, it=20):
    for i in range(it):
        lab = np.argmin(np.abs(x[:, None] - m[None]), 1)
        yeni = np.array([x[lab == k].mean() for k in range(len(m))])
        print(f"  it {i+1}: merkezler {m} -> {yeni.round(2)}, atamalar {lab}")
        if np.allclose(yeni, m): break
        m = yeni
    return m, lab, ((x - m[lab])**2).sum()
for m0 in ([2., 25.], [3., 10.]):
    print("başlangıç", m0); m, lab, inertia = kmeans1d(x, np.array(m0)); print("  inertia =", round(inertia, 1))

## 3. k-means: k seçimi (dirsek + silüet)

In [ ]:
Ks = range(2, 11); inertia, sil = [], []
for k in Ks:
    km = KMeans(k, n_init=10, random_state=0).fit(Z); inertia.append(km.inertia_); sil.append(silhouette_score(Z, km.labels_))
    print(f"k={k:2d}  inertia={km.inertia_:7.0f}  silüet={sil[-1]:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3)); ax[0].plot(list(Ks), inertia, "o-"); ax[0].set_title("dirsek"); ax[1].plot(list(Ks), sil, "s-"); ax[1].set_title("silüet")
for a in ax: a.set_xlabel("k"); a.grid(alpha=.3)
plt.show()

**Soru:** Dirsek nerede? Silüet hangi k'yı öneriyor? İkisi uyuşuyor mu?

## 4. Küme merkezlerini yorumlamak

In [ ]:
K = 5
km = KMeans(K, n_init=10, random_state=0).fit(Z); etiket = km.labels_
merkez = sc.inverse_transform(km.cluster_centers_)        # normalize-profil uzayına geri
plt.figure(figsize=(9, 4))
for k in range(K): plt.plot(h, merkez[k], lw=2.2, label=f"küme {k} (n={np.sum(etiket==k)})")
plt.xlabel("saat"); plt.ylabel("normalize yük"); plt.legend(); plt.grid(alpha=.3); plt.show()

**Görev:** Her kümeye bir isim verin (ör. 'ofis: 8-18 düz', 'konut: akşam tepesi', 'sanayi: 3 vardiya sabit', …) ve gerekçenizi yazın.

In [ ]:
# Ham kW büyüklüğü kümelerde nasıl dağılmış? (şekil ile büyüklük bağımsız mı?)
print(pd.DataFrame({"kume": etiket, "ort_kW": P.mean(1)}).groupby("kume").ort_kW.describe().round(1))

## 5. Silüet: örnek başına (Örnek 8.2)

In [ ]:
from sklearn.metrics import silhouette_samples
s = silhouette_samples(Z, etiket)
for k in range(K): print(f"küme {k}: ortalama silüet {s[etiket==k].mean():.3f}, negatif olan {np.sum(s[etiket==k] < 0)} abone")
print("En kötü yerleşmiş 5 abone:", ab.abone_id.values[np.argsort(s)[:5]])

## 6. Hiyerarşik kümeleme ve dendrogram

In [ ]:
L = linkage(Z, "ward")
plt.figure(figsize=(12, 4)); dendrogram(L, no_labels=True, color_threshold=14); plt.axhline(14, color="r", ls="--"); plt.ylabel("birleşme uzaklığı"); plt.show()
hk = AgglomerativeClustering(K, linkage="ward").fit_predict(Z)
print("k-means ile hiyerarşik uyumu (ARI):", round(adjusted_rand_score(etiket, hk), 3))

## 7. PCA: açıklanan varyans, skorlar, yükler (Örnek 8.3)

In [ ]:
C = np.array([[1, .8], [.8, 1]]); lam, V = np.linalg.eigh(C)
print("özdeğerler:", lam[::-1].round(2), " oranlar:", (lam/lam.sum())[::-1].round(2), " PC1 yönü:", V[:, -1].round(3))

In [ ]:
pca = PCA().fit(Z); oran = pca.explained_variance_ratio_
print("ilk 5 bileşen oranları:", oran[:5].round(3), " kümülatif:", np.cumsum(oran)[:5].round(3))
n90 = int(np.argmax(np.cumsum(oran) >= 0.9)) + 1; print("%90 için gereken bileşen:", n90)
T = pca.transform(Z)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].bar(range(1, 25), oran); ax[0].plot(range(1, 25), np.cumsum(oran), "o-", ms=3, color="C1"); ax[0].set_title("açıklanan varyans")
for k in range(K): ax[1].scatter(T[etiket==k, 0], T[etiket==k, 1], s=10, label=f"küme {k}")
ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2"); ax[1].legend(fontsize=7); ax[1].set_title("skorlar")
ax[2].plot(h, pca.components_[0], "o-", label="PC1"); ax[2].plot(h, pca.components_[1], "s-", label="PC2"); ax[2].axhline(0, color="k", lw=.5); ax[2].set_xlabel("saat"); ax[2].legend(); ax[2].set_title("yükler")
plt.tight_layout(); plt.show()

**Soru:** PC1'in yükleri hangi saatlerde pozitif, hangilerinde negatif? PC1 ekseni ne anlama geliyor? Hangi küme PC1'in hangi ucunda?

## 8. Kontrol: gerçek abone tipleriyle karşılaştırma

Gerçek tipler `abone_profilleri_etiket.csv` dosyasında (algoritma görmedi). Küme–tip çapraz tablosu ve ARI (1 = mükemmel uyum, 0 = rastgele).

In [ ]:
et = pd.read_csv("abone_profilleri_etiket.csv").set_index("abone_id").loc[ab.abone_id, "gercek_tip"].values
print(pd.crosstab(pd.Series(etiket, name="küme"), pd.Series(et, name="gerçek tip")))
print("ARI:", round(adjusted_rand_score(et, etiket), 3))

## 9. Alıştırmalar

**Alıştırma 1.** k-means'i normalize etmeden ham `P` üzerinde (standartlaştırılmış) çalıştırın; kümeler abone büyüklüğüne mi, şekline mi göre oluşuyor? Küme başına ortalama kW tablosuyla gösterin.

In [ ]:
# Alıştırma 1

**Alıştırma 2.** `n_init=1` ile k-means'i 10 farklı `random_state` ile çalıştırın; inertia değerlerinin dağılımını yazdırın. En iyi ile en kötü arasındaki fark ne kadar? n_init=10 sonucuyla karşılaştırın.

In [ ]:
# Alıştırma 2

**Alıştırma 3.** Hiyerarşik kümelemeyi 'single', 'complete', 'average', 'ward' bağlantılarıyla çalıştırıp her birinin ARI'sini (gerçek tiple) ve küme büyüklüklerini karşılaştırın. 'single' neden zincirleme (bir dev küme + tekiller) üretir?

In [ ]:
# Alıştırma 3

**Alıştırma 4.** PCA ile 24 boyutu 3 bileşene indirip `inverse_transform` ile geri dönüştürün; bir abone için orijinal ve yeniden oluşturulmuş profili çizin. Yeniden oluşturma hatasını (RMSE) bileşen sayısı 1, 2, 3, 5, 10 için hesaplayın.

In [ ]:
# Alıştırma 4

**Alıştırma 5.** PCA'yı 'gürültü giderici' olarak kullanın: profillere σ = 0.2 gürültü ekleyin, temiz veride öğrenilmiş 8 bileşenle geri oluşturun, orijinale olan hatayı gürültülü profilin hatasıyla karşılaştırın. Bileşen sayısını 2, 4, 8, 16 yapınca ne olur?

In [ ]:
# Alıştırma 5